In [1]:
!pip install braindecode

# Functions

In [2]:
from scipy.signal import butter, sosfiltfilt

def bandpass_filter(data, low=7, high=30, fs=160):
    """Applies bandpass filter to EEG data (channels, time_points)."""
    sos = butter(4, [low, high], btype='bandpass', fs=fs, output='sos')
    return sosfiltfilt(sos, data, axis=-1)

def preprocess_signals(X, use_filter=False):
    """Optional bandpass filter + maintain shape (N, C, T, 1)."""
    X_proc = np.zeros_like(X)
    for i in tqdm(range(X.shape[0]), desc="Preprocessing"):
        trial = np.squeeze(X[i], axis=-1)  # (C, T)
        if use_filter:
            trial = bandpass_filter(trial)
        X_proc[i] = np.expand_dims(trial, axis=-1)
    return X_proc

def normalize_per_channel(X):
    """Z-score normalize each channel per trial."""
    X_norm = np.zeros_like(X, dtype=np.float32)
    for i in range(X.shape[0]):
        trial = np.squeeze(X[i], axis=-1)  # (C, T)
        trial = (trial - np.mean(trial, axis=1, keepdims=True)) / np.std(trial, axis=1, keepdims=True)
        X_norm[i] = np.expand_dims(trial, axis=-1)
    return X_norm


# Mounting to G-Drive

In [3]:
# mount to google drive
import os
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation


# Importing Data

In [4]:
import pandas as pd
# import training data

# path to the training file pkl
train_df_path = '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/physionetdata/eegmmidb_train_df.pkl'
val_df_path = '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/physionetdata/eegmmidb_val_df.pkl'

In [7]:
from EEGNoiseAndTimeTransformations import apply_time_shift, apply_gaussian_noise, plot_single_eeg_sample, dataloader_to_numpy
from GANTraining.EEGMMIDBDatasetLoaderV2 import EEGMMIDBDataset
from torch.utils.data import DataLoader

train_dataset = EEGMMIDBDataset(pickle_path=train_df_path, purpose='eegnet', onehot=True)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = EEGMMIDBDataset(pickle_path=val_df_path, purpose='eegnet', onehot=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

## GAN Imports

In [4]:
from torch.utils.data import Dataset, DataLoader
import torch
from GANTraining.GANModels import *
import numpy as np
import os

In [5]:
# Define model paths
GAN_model_paths = {
    'left_hand': '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/GANTraining/logs/GAN_left_hand_exp_2025_07_06_20_28_14/Model/checkpoint_epoch111',
    'right_hand': '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/GANTraining/logs/GAN_right_hand_exp_2025_07_06_20_53_07/Model/checkpoint_epoch113',
    'both_hands': '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/GANTraining/logs/GAN_both_hands_exp_2025_07_06_21_18_14/Model/checkpoint_epoch111',
    'both_feet': '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/GANTraining/logs/GAN_both_feet_exp_2025_07_06_21_43_17/Model/checkpoint_epoch111',
    'rest_model': '/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/GANTraining/logs/GAN_rest_exp_2025_07_07_01_53_42/Model/checkpoint_epoch28'
}

In [6]:
!ls

arl-eegmodels			      EEGNoiseAndTimeTransformations.py
EEGMMIDBDatasetLoaderV2.ipynb	      GANTraining
EEGMMIDBDatasetLoaderV2.py	      physionetdata
EEGMMIDBDatasetLoaderV3.ipynb	      PrelimanaryEEGNetTraining.ipynb
EEGMMIDBSignalV6.ipynb		      __pycache__
EEGModels.py			      saved_eegnet_models
eegnet_eval_results.csv		      StackedDataAugmentationTraining.ipynb
EEGNoiseAndTimeTransformations.ipynb


In [7]:
%cd GANTraining

/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation/GANTraining


In [8]:
from GANTraining.SyntheticGANDataset import SyntheticGANEEGDataset

In [9]:
%cd ..

/content/drive/MyDrive/Colab Notebooks/StackedDataAugmentation


# Data Insight of Training Data

In [24]:
import numpy as np
# loading data
X_train, y_train = dataloader_to_numpy(train_loader)

X_val, y_val = dataloader_to_numpy(val_loader)

# convert one-hot label to class indices
y_train_indices = np.argmax(y_train, axis=1)
y_val_indices = np.argmax(y_val, axis=1)

# get distribution
unique, counts = np.unique(y_train_indices, return_counts=True)

# print distribution
for label, count in zip(unique, counts):
  print(f'Class {label}, Count: {count}')

# get data shape information
print(X_train.shape)            # (8222, channels, time_points)
print(X_train[0].shape)         # should be (channels, time_points)

Class 0, Count: 1656
Class 1, Count: 1629
Class 2, Count: 1644
Class 3, Count: 1641
Class 4, Count: 1652
(8222, 64, 640, 1)
(64, 640, 1)


In [25]:
# converting data to torch tensors
X_train_torch = torch.tensor(X_train.squeeze(-1), dtype=torch.float32)
y_train_torch = torch.tensor(y_train_indices, dtype=torch.long)

X_val_torch = torch.tensor(X_val.squeeze(-1), dtype=torch.float32)
y_val_torch = torch.tensor(y_val_indices, dtype=torch.long)

print(X_train_torch.shape, X_train_torch.dtype)  # should be (N, 64, 640) float32
print(y_train_torch.shape, y_train_torch.dtype)  # should be (N,) long

torch.Size([8222, 64, 640]) torch.float32
torch.Size([8222]) torch.int64


# Preprocessing Signals

In [15]:
import numpy as np
from tqdm import tqdm

# Assume shape is (N, C, T)
X_train_filtered = np.zeros_like(X_train)

for i in tqdm(range(X_train.shape[0])):
    # Remove last dim → (64, 640)
    trial = np.squeeze(X_train[i], axis=-1)

    # Apply bandpass filter
    filtered = bandpass_filter(trial)  # still shape (64, 640)

    # Add back trailing dim for model compatibility → (64, 640, 1)
    X_train_filtered[i] = np.expand_dims(filtered, axis=-1)

100%|██████████| 8222/8222 [00:26<00:00, 315.24it/s]


# Prelimanary Training of Models

## EEGNet Model

In [16]:
from EEGModels import EEGNet

In [17]:
eegnet_model = EEGNet(nb_classes=5,    # num of classes
                  Chans=64,       # num of channels
                  Samples=640,     # seq_len
                  )
eegnet_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

# train model
eegnet_model.fit(X_train, y_train, epochs=100, batch_size=64)

# evaluate val data
eegnet_model.evaluate(X_val, y_val)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 16s 61ms/step - accuracy: 0.2873 - loss: 1.5733
Epoch 2/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.4315 - loss: 1.4132
Epoch 3/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.4598 - loss: 1.3683
Epoch 4/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.4738 - loss: 1.3347
Epoch 5/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.4837 - loss: 1.3265
Epoch 6/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.4737 - loss: 1.3385
Epoch 7/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.4904 - loss: 1.3011
Epoch 8/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.4816 - loss: 1.3171
Epoch 9/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.5110 - loss: 1.2959
Epoch 10/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.4867 - loss: 1.3030
Epoch 11/100
129/129 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.5040 - loss: 1.2861
Epoch 12/100
129/129 ━━━━━━━━

KeyboardInterrupt: 

## ShallowNetModel

In [26]:
from braindecode.models import ShallowFBCSPNet
from braindecode import EEGClassifier
import torch
import numpy as np




# making shallow net
shallownet_model = ShallowFBCSPNet(
    n_outputs=5,
    n_chans=64,
    n_times=640
)

# Wrap model for sklearn-like API
clf = EEGClassifier(
    shallownet_model,
    criterion=torch.nn.CrossEntropyLoss,
    optimizer=torch.optim.Adam,
    optimizer__lr=0.001,
    batch_size=64,
    max_epochs=100,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

# Fit
clf.fit(X_train_torch, y_train_torch)

# get accuracy of model
clf.score(X_val_torch, y_val_torch)

  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        7.7392       0.2061        1.6807  2.2127
      2        3.9287       0.2006        2.0028  2.0808
      3        3.3876       0.2024        1.9608  2.0970
      4        3.0033       0.2030        3.0135  2.0966
      5        2.7663       0.2012        2.3794  2.1194
      6        2.5058       0.2474        1.7754  2.1079
      7        2.2557       0.2480        1.6493  2.1112


0.22601530311948204

## Deep4Net

In [23]:
from braindecode.models import Deep4Net
from braindecode import EEGClassifier
import torch

# builds deepconv model
deepconv_model = Deep4Net(
    n_outputs=5,    # number of classes
    n_chans=64,     # EEG channels
    n_times=640,
    final_conv_length='auto'
)

# wrap EEGClassifier
clf = EEGClassifier(
    deepconv_model,
    criterion=torch.nn.CrossEntropyLoss,   # Loss function
    optimizer=torch.optim.Adam,            # Optimizer
    optimizer__lr=0.001,                    # Learning rate
    batch_size=64,
    max_epochs=100,
    train_split=None,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

# train
clf.fit(X_train_torch, y_train_torch)

# evaluate
val_acc = clf.score(X_val_torch, y_val_torch)
print(f"Deep4Net Validation Accuracy: {val_acc*100:.2f}%")

torch.Size([8222, 64, 640]) torch.float32
torch.Size([8222]) torch.int64
  epoch    train_loss     dur
-------  ------------  ------
      1        1.7433  1.6783
      2        1.4526  1.4740
      3        1.3762  1.4771
      4        1.3429  1.4751
      5        1.3122  1.4935
      6        1.2901  1.4876
      7        1.2662  1.4924
      8        1.2540  1.5436
      9        1.2480  1.4983
     10        1.2337  1.4807
     11        1.2168  1.4834
     12        1.2069  1.4877
     13        1.1980  1.4887
     14        1.1893  1.4884
     15        1.1765  1.4911
     16        1.1733  1.5465
     17        1.1561  1.5161
     18        1.1441  1.4980
     19        1.1294  1.4995
     20        1.1193  1.4959
     21        1.1053  1.4937
     22        1.1016  1.4835
     23        1.1005  1.4821
     24        1.0914  1.5798
     25        1.0761  1.5026
     26        1.0671  1.4922
     27        1.0695  1.4757
     28        1.0505  1.4628
     29        1.0399  1.47

# Accuracy Notes
- EEGNet Without Preprocessing (BP Filter) -> 44.81% accuracy

- EEGNet with preprocessing -> 42.62

# Evaluating Performance

In [ ]:
# looking at confusion matrix
